# 27 — The Orbitrap: orbital trapping, image current, and the Fourier route to m/z

## Background

In 1923 Kingdon showed that ions attracted to a thin charged wire can be
trapped on stable orbits around it. Knight (1981) added shaped end
electrodes to superimpose an axial potential well, and Makarov (2000)
perfected the idea into the **Orbitrap**: a spindle-shaped central
electrode inside a barrel-shaped outer electrode, machined so that the
vacuum between them carries the **quadro-logarithmic** potential

$$U(r,z) = \frac{k}{2}\left(z^2 - \frac{r^2 - R_2^2}{2}\right) + \frac{k}{2}\,R_m^2 \ln\!\frac{r}{R_2} + C$$

Two properties make this field a mass analyzer:

1. **Separability** — the axial force $F_z = -q\,\partial U/\partial z = -qkz$
   is a pure Hooke's-law restoring force, *independent of* $r$ and of the
   orbital motion. Every trapped ion is a harmonic oscillator along the
   trap axis with angular frequency

   $$\omega_z = \sqrt{\frac{q\,k}{m}} \quad\Longleftrightarrow\quad \frac{m}{z}\,[\mathrm{Th}] = \frac{e\,k}{u\,(2\pi f_z)^2}$$

   where $e$ is the elementary charge and $u$ the atomic mass unit. The
   frequency depends on **nothing but m/z** — not on the oscillation
   amplitude, not on the orbital radius. That amplitude-independence
   (*isochronism*) is the entire measurement principle.

2. **Radial confinement** — an ion injected tangentially with angular
   momentum $L$ sees an effective radial potential with a minimum, so it
   circulates between $r_{min}$ and $r_{max}$ indefinitely under UHV.

The electrodes themselves are equipotential surfaces of $U$, truncated at
finite length:

$$z_{1,2}(r) = \sqrt{\frac{r^2 - R_{1,2}^2}{2} + R_m^2\,\ln\!\frac{R_{1,2}}{r}}$$

with $R_1$ the central-spindle radius and $R_2$ the outer-barrel radius
at $z=0$.

**Detection** is by *image current*: the outer electrode is split into
two halves, and the axial oscillation of the ion cloud induces a
differential charge on them. The recorded transient of a mixture is a sum
of sinusoids — an *interferogram* — and a Fourier transform recovers each
species' frequency, hence its m/z.

**Literature anchors.** Makarov, *Anal. Chem.* 2000; Grinfeld,
Monastyrskiy & Makarov, *Microsc. Microanal.* 2015 (the operating point
used here: 10 mm outer electrode at ground, central electrode at
−3.5 kV); Gall/Golikov 1986 for the ideal-field dynamics.

**Claim under test.** The deck `examples/orbitrap_rz.json` — two
truncated quadro-logarithmic equipotentials, solved on a raster, no
injection slot — traps tangentially injected ions on stable orbits whose
axial frequency measures m/z: one known species calibrates the trap, and
a Fourier transform of a mixture's summed axial signal recovers every
component's m/z from a single transient.


## From frequency to m/z — the two routes used below

**Route 1: first principles.** With the deck's geometry the field
curvature is
$$k = \frac{2\,V_c}{\tfrac{R_1^2 - R_2^2}{2} + R_m^2 \ln(R_2/R_1)}$$
so a measured $f_z$ gives m/z directly via
$m/z = e\,k / \big(u\,(2\pi f_z)^2\big)$. On a rasterized solve this
carries the pitch-dependent surface-placement bias of the field
(measured on this deck: $-0.74\%$ in $f$ at $h=0.05$ mm, halving with
$h$), which enters m/z **doubled**, since $m/z \propto f^{-2}$.

**Route 2: calibration (what real instruments do).** For any fixed trap
and charge state, $f^2\cdot(m/z) \equiv C$ is a constant. One known
calibrant measured in the *same* field gives $C = f_{cal}^2 (m/z)_{cal}$,
and every unknown follows as
$$ (m/z)_i = C / f_i^{\,2}. $$
Calibration cancels the raster bias to first order because the calibrant
and the unknowns fly in the *same imperfect field*. The residual is the
field's **anharmonicity** (amplitude dependence of $f$), measured on this
deck as $\sim$2600–8100 ppm depending on pitch — the notebook states it
where it matters.


In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (charter: stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
# --- Named parameters (the ONLY cell with physics choices) -------------
# Deck: the promoted example; geometry, voltages, solver pitch, default
# source, and integration settings all live IN the deck, not here.
DECK_PATH = "../examples/orbitrap_rz.json"

ZC_MM = 12.0          # trap centre along the solver x axis [mm] — the deck
                      # is built symmetric about x = 12 (domain 0..24)
CAL_MZ = 500.0        # calibrant m/z: the deck's default ion; its
                      # measured frequency sets the calibration constant C
MIX_MZ = [322.048, 622.029, 922.010, 1221.991]
                      # mixture m/z values: ESI-tune-like ladder spanning ~4x in
                      # mass, so the interferogram shows well-separated
                      # frequencies (f ~ 1/sqrt(m/z))
T_MIX_US = 200.0      # mixture record length [us]: FFT line spacing is
                      # 1/T = 5 kHz, far below the ~50-200 kHz species
                      # separations, so every component resolves cleanly
AX_CH = "x"           # axial coordinate channel name (trap axis is x in
                      # r-z decks); trajectories are read BY NAME (H9)


## Load the instrument and show what will be flown

The deck is loaded exactly as a user would load it; `build_run` solves
(or cache-hits) the field. The geometry figures below are rendered from
the **solver's own mask and field** — what you see is what the ions fly
in. The instrument is a body of revolution, so it is shown both as the
r-z section (with the solved potential) and revolved, multi-axis.

In [ ]:
import sys, time
sys.path.insert(0, "..")
import numpy as np
from ion_gym.io.sim_spec import SimSpec
from ion_gym.io.deck_params import describe_deck
from ion_gym.physics.sim_build import build_run
from ion_gym.viz.viz_core import scene_from_simspec, render_mpl

spec = SimSpec.from_json(DECK_PATH)
describe_deck(spec)   # everything inherited from the deck, verbatim
t0 = time.time()
model, fly, cols, births = build_run(spec)
print(f"field ready in {time.time()-t0:.1f} s | grid {model.A.shape} at "
      f"{spec.geometry.mm_per_gu} mm/gu")
ICOL = {n: i for i, n in enumerate(cols)}   # named column access (H9)


In [ ]:
# r-z section with the solved potential: the two truncated
# quadro-logarithmic equipotential surfaces and the field between them.
OP = (f"R1 6 mm @ {spec.geometry.electrodes[0].dc:g} V, R2 10 mm @ GND, "
      f"Rm 14.142 mm, |z| <= 12 mm, pitch {spec.geometry.mm_per_gu} mm")
fig = render_mpl(scene_from_simspec(spec, model, field="phi",
    title=f"Orbitrap r-z section -- solved potential | {OP}"),
    layout="column")
# sized + backend-proof display (natural-size figures
# did not fit one screen, and a bare trailing `fig` fails to render on
# kernels without the inline hook)
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt
_buf = _io.BytesIO()
fig.savefig(_buf, format="png", dpi=100, bbox_inches="tight")
_display(_PNG(_buf.getvalue(), height=500))
_plt.close(fig)


In [ ]:
# The same instrument revolved (it is a body of revolution): xy, zx
# and zy projections -- multi-axis, per the 3-D representation rule.
fig = render_mpl(scene_from_simspec(spec, model, field="phi", revolve=72,
    title=f"Orbitrap revolved, 3 views -- x = trap axis | {OP}"),
    layout="column")
# sized + backend-proof display (natural-size figures
# did not fit one screen, and a bare trailing `fig` fails to render on
# kernels without the inline hook)
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt
_buf = _io.BytesIO()
fig.savefig(_buf, format="png", dpi=100, bbox_inches="tight")
_display(_PNG(_buf.getvalue(), height=780))
_plt.close(fig)


## One ion first: fly the calibrant and see the motion

The deck's default source births a single m/z 500 ion **on** the
populated orbit of the 2015 paper — r = 6.5 mm, axial offset 4.8 mm,
purely tangential at the circular-orbit kinetic energy. If the field is
right it must (a) hold the ion at nearly constant radius and (b)
oscillate it axially at a single frequency. The trajectory is overlaid on
the section so the classic Orbitrap "weave" is visible.

In [ ]:
tr_cal, st_cal = fly(0)
assert st_cal["kind"] == 2, f"calibrant not held: fate {st_cal['kind']}"
t = tr_cal[:, ICOL["t"]]
ax = tr_cal[:, ICOL[AX_CH]] - ZC_MM
rr = np.hypot(tr_cal[:, ICOL["y"]], tr_cal[:, ICOL["z"]])
print(f"held {t[-1]:.0f} us | r = {rr.mean():.3f} +/- {rr.std():.3f} mm "
      f"| axial amplitude {np.abs(ax).max():.2f} mm")
fig = render_mpl(scene_from_simspec(spec, model, field="phi",
    trajs=[tr_cal], fates=[st_cal["kind"]],
    title=(f"Calibrant flight, m/z {CAL_MZ:g} -- r {rr.mean():.2f} mm, "
           f"Z_amp {np.abs(ax).max():.1f} mm | {OP}")), layout="column")
# sized + backend-proof display (natural-size figures
# did not fit one screen, and a bare trailing `fig` fails to render on
# kernels without the inline hook)
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt
_buf = _io.BytesIO()
fig.savefig(_buf, format="png", dpi=100, bbox_inches="tight")
_display(_PNG(_buf.getvalue(), height=500))
_plt.close(fig)


## Measure the axial frequency and turn it into m/z

Zero crossings of the axial coordinate, linearly interpolated between
the 4 ns samples, time every half-period; the count over the full record
gives the frequency to ~10 ppm. Route 1 converts it with the ideal-field
k and shows the raster bias honestly; Route 2 uses the same measurement
to set the calibration constant C used for the mixture.

In [ ]:
E_C = 1.602176634e-19      # elementary charge [C]
AMU = 1.66053906660e-27    # atomic mass unit [kg]

def f_axial_khz(tr):
    '''Axial frequency from interpolated zero crossings [kHz].'''
    tt = tr[:, ICOL["t"]]
    aa = tr[:, ICOL[AX_CH]] - ZC_MM
    idx = np.where(np.diff(np.sign(aa)) != 0)[0]
    tc = tt[idx] - aa[idx] * (tt[idx+1] - tt[idx]) / (aa[idx+1] - aa[idx])
    return (len(tc) - 1) / 2.0 / (tc[-1] - tc[0]) * 1e3

# Route 1: ideal-field k from the deck's operating point
R1, R2, RM, VC = 6e-3, 10e-3, 14.142e-3, 3500.0        # [m], [V]
K = 2 * VC / ((R1**2 - R2**2) / 2 + RM**2 * np.log(R2 / R1))  # [V/m^2]
f_cal = f_axial_khz(tr_cal)                             # [kHz]
mz_route1 = E_C * K / (AMU * (2 * np.pi * f_cal * 1e3) ** 2)
print(f"measured f_axial = {f_cal:.2f} kHz")
print(f"Route 1 (ideal k = {K:.3e} V/m^2): m/z = {mz_route1:.1f} Th "
      f"({1e2*(mz_route1/CAL_MZ - 1):+.2f}% -- the raster bias, doubled "
      f"because m/z ~ f^-2)")

# Route 2: calibration constant from the SAME field
C_CAL = f_cal**2 * CAL_MZ            # [kHz^2 * Th]
print(f"Route 2: C = f^2 (m/z) = {C_CAL:.4e} kHz^2 Th "
      f"-- every unknown below is m/z = C / f^2")


## A mixture: the interferogram and its solution

Each mixture component is flown from the same birth conditions (only m/z
changes — voltages, ions, and integration are reweights of the cached
field, so no re-solve happens). The detected quantity in a real Orbitrap
is the differential image charge on the two outer-electrode halves,
which for small axial excursion is proportional to the ion's axial
coordinate; the **sum of axial coordinates** over the cloud is therefore
the standard pedagogical proxy for the transient. Summing the four
single-ion records gives the mixture interferogram a student would see
on the detector.

In [ ]:
mix_tr = []
for mz in MIX_MZ:
    s = SimSpec.from_json(DECK_PATH)
    s.source.mz_list = [mz]              # declared field: guard-checked
    s.integration.t_max_us = T_MIX_US    # longer record for FFT lines
    _m, _fly, _c, _b = build_run(s)      # cache hit: geometry unchanged
    tr, st = _fly(0)
    assert st["kind"] == 2, f"m/z {mz} not held: fate {st['kind']}"
    mix_tr.append(tr)
    print(f"m/z {mz:8.3f}: held {tr[-1, ICOL['t']]:.0f} us, "
          f"f = {f_axial_khz(tr):.2f} kHz")


In [ ]:
# Time-domain interferogram: the summed axial signal (image-charge
# proxy). All records share the deck's clock, so they sum sample-wise.
import matplotlib.pyplot as plt
tt = mix_tr[0][:, ICOL["t"]]
S = np.sum([tr[:, ICOL[AX_CH]] - ZC_MM for tr in mix_tr], axis=0)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
for a, sl, lab in [(axs[0], slice(None), f"full {T_MIX_US:g} us record"),
                   (axs[1], slice(0, np.searchsorted(tt, 20.0)),
                    "first 20 us -- individual beats visible")]:
    a.plot(tt[sl], S[sl], lw=0.5)
    a.set_xlabel("t (us)"); a.set_ylabel("sum of axial positions (mm)")
    a.set_title(lab, fontsize=9)
fig.suptitle(f"Mixture interferogram -- {len(MIX_MZ)} species "
             f"{MIX_MZ} Th, 1 ion each | {OP}", fontsize=9)
fig.tight_layout()


In [ ]:
# Frequency domain: FFT magnitude of the interferogram, then the SAME
# spectrum replotted on the m/z axis via the calibration m/z = C / f^2.
dt_us = tt[1] - tt[0]
F = np.fft.rfft(S * np.hanning(len(S)))
fk = np.fft.rfftfreq(len(S), d=dt_us) * 1e3          # [kHz]
mag = np.abs(F)
band = (fk > 200) & (fk < 1200)                       # ion band of interest

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].plot(fk[band], mag[band], lw=0.8)
axs[0].set_xlabel("f (kHz)"); axs[0].set_ylabel("|FFT| (arb)")
axs[0].set_title("frequency domain", fontsize=9)
mz_axis = C_CAL / fk[band] ** 2
axs[1].plot(mz_axis, mag[band], lw=0.8)
axs[1].set_xlim(250, 1400)
axs[1].set_xlabel("m/z  [= C / f^2]"); axs[1].set_ylabel("|FFT| (arb)")
axs[1].set_title("same spectrum, m/z domain", fontsize=9)
for mz in MIX_MZ:
    axs[1].axvline(mz, color="0.7", lw=0.6, zorder=0)
fig.suptitle(f"Mixture solved -- Hann window, T = {T_MIX_US:g} us, "
             f"line spacing {1e3/T_MIX_US:.0f} kHz | grey: true m/z",
             fontsize=9)
fig.tight_layout()

# Peak-pick and report recovered m/z against truth
pk = []
for mz_true in MIX_MZ:
    f_pred = np.sqrt(C_CAL / mz_true)
    w = (fk > 0.9 * f_pred) & (fk < 1.1 * f_pred)
    f_meas = fk[w][np.argmax(mag[w])]
    # 3-point parabolic interpolation around the bin peak
    i = np.argmax(mag[w]); j = np.where(w)[0][i]
    if 0 < j < len(mag) - 1:
        a, b, c = mag[j-1], mag[j], mag[j+1]
        f_meas = fk[j] + 0.5 * (a - c) / (a - 2*b + c) * (fk[1] - fk[0])
    mz_meas = C_CAL / f_meas**2
    pk.append((mz_true, f_meas, mz_meas, 1e6 * (mz_meas / mz_true - 1)))
print(f"{'true m/z':>10} {'f (kHz)':>9} {'recovered':>10} {'ppm':>8}")
for row in pk:
    print(f"{row[0]:10.3f} {row[1]:9.2f} {row[2]:10.3f} {row[3]:8.0f}")


## Read-out

- **Section and 3-view figures**: the instrument actually flown — two
  truncated quadro-logarithmic equipotentials with the solved potential
  between them. *Falsifying picture*: contours crowding or kinking near
  the electrodes (raster failure), or field leaking through the metal.
- **Calibrant flight**: constant-radius orbit weaving axially — orbital
  trapping plus harmonic axial motion in one picture. *Falsifying
  picture*: fate ≠ held (ion lost to an electrode) or a visibly
  drifting radius (wrong tangential energy or a non-separable field).
- **Route 1 vs Route 2**: the ideal-k conversion lands ~1.5% high on a
  known ion — twice the −0.74% frequency bias of the h = 0.05 mm
  raster, exactly as m/z ∝ f⁻² predicts. Calibration against one known
  species absorbs that bias; this is why real instruments calibrate.
- **Interferogram**: four incommensurate sinusoids beating — no single
  species is legible by eye, which is the point.
- **m/z-domain spectrum**: four resolved peaks on the true m/z lines
  (grey). The recovered values land within the deck's anharmonicity
  budget (thousands of ppm at this pitch — a *field-quality* limit, not
  a method limit; the h-ladder shows it shrinking ~O(h)).
  *Falsifying picture*: peaks missing, split, or displaced by more than
  that budget, or frequencies failing the f ∝ 1/√(m/z) ordering.

**What was established**: a JSON-only Orbitrap deck — no device-specific
code anywhere in the core — traps ions stably, its axial frequency
measures m/z through one-point calibration, and a single Fourier
transform of the summed transient solves a four-component mixture.
